## **1. Environment Setup and Configuration**
Setting up relative paths for the dataset and output directories to ensure repository portability.

In [1]:
import os
import glob
import pickle
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Define dynamic relative paths assuming the notebook runs inside the 'notebooks/' directory
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATASET_PATH = os.path.join(BASE_DIR, "Dataset")
RESULTS_DIR = os.path.join(BASE_DIR, "results")

# Ensure the results directory exists for later metadata exports
os.makedirs(RESULTS_DIR, exist_ok=True)

# Detect hardware accelerator
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("Environment Configuration:")
print(f"Base Directory: {BASE_DIR}")
print(f"Dataset Path: {DATASET_PATH}")
print(f"Results Directory: {RESULTS_DIR}")
print(f"Active Device: {device}")

Environment Configuration:
Base Directory: C:\Users\javed\Home\Deepfake-Detection
Dataset Path: C:\Users\javed\Home\Deepfake-Detection\Dataset
Results Directory: C:\Users\javed\Home\Deepfake-Detection\results
Active Device: cuda


## **2. Data Integrity and Verification**
Scanning the `./Dataset` directory to ensure the correct train/val/test structure and verifying image files to catch any corrupted data before loading.

In [2]:
def verify_dataset(dataset_path, sample_size=50):

    #Consolidated function to verify directory structure, count images per class, and check a sample of images for file corruption.
    splits = ['train', 'validation', 'test']
    classes = ['real', 'fake']
    
    print("Dataset Verification Results:")
    
    if not os.path.exists(dataset_path):
        print(f"Error: Dataset directory not found at {dataset_path}")
        return False
        
    total_images = 0
    corrupted_images = 0
    
    for split in splits:
        split_path = os.path.join(dataset_path, split)
        if not os.path.exists(split_path):
            print(f"Missing directory: {split}")
            continue
            
        for cls in classes:
            cls_path = os.path.join(split_path, cls)
            if not os.path.exists(cls_path):
                print(f"Missing directory: {split}/{cls}")
                continue
                
            # Filter valid image extensions
            images = glob.glob(os.path.join(cls_path, "*.jpg")) + \
                     glob.glob(os.path.join(cls_path, "*.jpeg")) + \
                     glob.glob(os.path.join(cls_path, "*.png"))
            
            count = len(images)
            total_images += count
            print(f"{split}/{cls}: {count} images")
            
            # Open a small sample to verify file headers are intact
            sample = images[:sample_size]
            for img_path in sample:
                try:
                    with Image.open(img_path) as img:
                        img.verify()
                except Exception:
                    corrupted_images += 1
                    
    print(f"Total Images Found: {total_images}")
    print(f"Corrupted Images in Sample: {corrupted_images}")
    
    return total_images > 0 and corrupted_images == 0

# Execute the verification pipeline
is_dataset_valid = verify_dataset(DATASET_PATH)

Dataset Verification Results:
train/real: 70001 images
train/fake: 70001 images
validation/real: 19787 images
validation/fake: 19641 images
test/real: 5413 images
test/fake: 5492 images
Total Images Found: 190335
Corrupted Images in Sample: 0


## **3. Transformations and Dataset Class**
Defining the standard preprocessing pipeline (resizing, normalization) and training augmentations (flips, color jitter) alongside the custom PyTorch Dataset class.

In [3]:
# Standardized input dimensions for Custom CNN and pre-trained backbones
INPUT_SIZE = 224

# Training pipeline with augmentation to improve model generalization
train_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Validation/Test pipeline strictly requires resizing and normalization
val_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class DeepfakeDataset(Dataset):
    """
    Custom PyTorch Dataset for binary image classification.
    Maps 'fake' -> 0 and 'real' -> 1.
    """
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.images = []
        self.labels = []
        self.class_to_label = {'fake': 0, 'real': 1}
        
        for class_name, label in self.class_to_label.items():
            class_path = os.path.join(root_dir, class_name)
            if os.path.exists(class_path):
                for img_file in os.listdir(class_path):
                    if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                        self.images.append(os.path.join(class_path, img_file))
                        self.labels.append(label)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception:
            # Safely skip unreadable files and load the next available image
            return self.__getitem__((idx + 1) % len(self.images))
            
        if self.transform:
            image = self.transform(image)
            
        return image, label

print("Transformations and Dataset Class Configured:")
print(f"Resolution: {INPUT_SIZE}x{INPUT_SIZE}")
print(f"Class Mapping: {{'fake': 0, 'real': 1}}")

Transformations and Dataset Class Configured:
Resolution: 224x224
Class Mapping: {'fake': 0, 'real': 1}


## **4. DataLoader Initialization and Metadata Export**
Creating batched data loaders for training, validation, and testing. Exporting dataset metadata for seamless integration with the subsequent training phase.

In [4]:
BATCH_SIZE = 64

# Initialize Datasets
train_dataset = DeepfakeDataset(os.path.join(DATASET_PATH, 'train'), transform=train_transform)
val_dataset = DeepfakeDataset(os.path.join(DATASET_PATH, 'validation'), transform=val_transform)
test_dataset = DeepfakeDataset(os.path.join(DATASET_PATH, 'test'), transform=val_transform)

# Initialize DataLoaders
# pin_memory=True speeds up data transfer to CUDA
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

# Export Metadata for Phase 2 (Model Training)
dataset_info = {
    'train_size': len(train_dataset),
    'val_size': len(val_dataset),
    'test_size': len(test_dataset),
    'batch_size': BATCH_SIZE,
    'input_size': INPUT_SIZE,
    'class_names': ['fake', 'real'],
    'num_classes': 2
}

info_path = os.path.join(RESULTS_DIR, 'dataset_info.pkl')
with open(info_path, 'wb') as f:
    pickle.dump(dataset_info, f)

print("DataLoaders Created and Metadata Exported:")
print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")
print(f"Metadata saved to: {info_path}")

DataLoaders Created and Metadata Exported:
Train batches: 2188
Validation batches: 617
Test batches: 171
Metadata saved to: C:\Users\javed\Home\Deepfake-Detection\results\dataset_info.pkl


### DataLoader Sanity Check
Fetching a single batch to verify tensor shapes and label mapping.

In [5]:
# Fetch a single batch to verify pipeline integrity
images, labels = next(iter(train_loader))

print("Batch Sanity Check:")
print(f"Images tensor shape: {images.shape}")
print(f"Labels tensor shape: {labels.shape}")
print(f"Sample labels (0=fake, 1=real): {labels[:5].tolist()}")

Batch Sanity Check:
Images tensor shape: torch.Size([64, 3, 224, 224])
Labels tensor shape: torch.Size([64])
Sample labels (0=fake, 1=real): [1, 1, 1, 0, 0]
